# 🧠 Milestone 3 — CNN from Scratch (PyTorch)
**Train a Convolutional Neural Network on Mel-Spectrograms**

### What you'll learn:
- PyTorch basics: Tensors, Dataset, DataLoader
- How to convert audio → 2D Mel-Spectrogram images
- How to build a CNN architecture from scratch
- Training loop with loss, optimizer, validation
- Full W&B experiment tracking

### Why CNN works for audio?
Mel-Spectrograms are 2D images (time × frequency). CNNs are great at finding local patterns in images, so they naturally capture rhythmic and timbral patterns in audio!

In [ ]:
!pip install librosa wandb -q
# PyTorch is pre-installed on Kaggle

In [ ]:
import os, random, warnings, time
import numpy as np
import pandas as pd
import librosa
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

BASE        = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUPS_DIR = f'{BASE}/mashups'
TEST_CSV    = f'{BASE}/test.csv'
GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
ROLL_NO     = 'YOUR_ROLL_NO'  # ⚠️ change this!

## 1️⃣ Audio → Mel-Spectrogram
We convert each audio file into a **fixed-size image** (128 × 256 pixels).
Each pixel = energy at a certain frequency at a certain time.

In [ ]:
# ── Hyperparameters (tune these!) ─────────────────────────────────────────────
CONFIG = {
    'sample_rate'  : 22050,
    'duration'     : 30,       # seconds to load per file
    'n_mels'       : 128,      # mel frequency bins (height of image)
    'n_fft'        : 2048,     # FFT window size
    'hop_length'   : 512,      # step between FFT windows
    'target_length': 256,      # fixed time frames (width of image)
    'batch_size'   : 32,
    'epochs'       : 30,
    'lr'           : 1e-3,
    'dropout'      : 0.3,
    'model'        : 'CNN_from_scratch',
}

def audio_to_melspec(path, cfg=CONFIG):
    """
    Load audio and return a normalized mel-spectrogram.
    Output shape: (1, n_mels, target_length) — single channel image.
    """
    try:
        y, sr = librosa.load(path, sr=cfg['sample_rate'], duration=cfg['duration'])
        if len(y) < sr:
            y = np.zeros(sr * cfg['duration'])
        mel    = librosa.feature.melspectrogram(
                    y=y, sr=sr, n_mels=cfg['n_mels'],
                    n_fft=cfg['n_fft'], hop_length=cfg['hop_length'])
        mel_db = librosa.power_to_db(mel, ref=np.max)  # convert to dB scale
        # Pad or crop to fixed width
        T = cfg['target_length']
        if mel_db.shape[1] < T:
            mel_db = np.pad(mel_db, ((0,0),(0, T - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :T]
        # Normalize to [0, 1]
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db[np.newaxis, :, :]  # add channel dim → (1, 128, 256)
    except Exception as e:
        return np.zeros((1, CONFIG['n_mels'], CONFIG['target_length']))

# Quick test
sample_path = f"{STEMS_DIR}/blues/{sorted(os.listdir(f'{STEMS_DIR}/blues'))[0]}/vocals.wav"
spec = audio_to_melspec(sample_path)
print(f'Mel-spec shape: {spec.shape}  (channels, freq_bins, time_frames)')

plt.figure(figsize=(10,3))
plt.imshow(spec[0], aspect='auto', origin='lower', cmap='inferno')
plt.colorbar(); plt.title('Mel-Spectrogram (blues/vocals)')
plt.xlabel('Time'); plt.ylabel('Mel Frequency')
plt.tight_layout(); plt.show()

## 2️⃣ Build Dataset
PyTorch `Dataset` is just a class with `__len__` and `__getitem__` — it tells the DataLoader how to fetch one sample.

In [ ]:
class AudioDataset(Dataset):
    """Dataset for training stems. Randomly picks ONE stem per song per call."""
    def __init__(self, file_list, label_list, augment=False):
        self.files   = file_list
        self.labels  = label_list
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path  = self.files[idx]
        label = self.labels[idx]
        mel   = audio_to_melspec(path)

        if self.augment:
            # Time masking — randomly zero out time frames
            t_start = random.randint(0, CONFIG['target_length'] - 30)
            mel[0, :, t_start:t_start+20] = 0
            # Frequency masking — randomly zero out freq bands
            f_start = random.randint(0, CONFIG['n_mels'] - 15)
            mel[0, f_start:f_start+10, :] = 0

        return torch.FloatTensor(mel), torch.tensor(label, dtype=torch.long)

class TestDataset(Dataset):
    """Dataset for test mashup files."""
    def __init__(self, file_list):
        self.files = file_list
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        mel = audio_to_melspec(self.files[idx])
        return torch.FloatTensor(mel)

print('✅ Dataset classes defined')